# Remove PDF password protection

Use this notebook to save an unencrypted copy of a PDF when you already know its password. Only process documents you own or are authorized to modify. This notebook does not guess or crack passwords.

The PDF is uploaded to this temporary Google Colab runtime. Do not use this workflow for confidential documents unless that is acceptable. The notebook does not mount Google Drive or print the password or document contents.

## Install pikepdf

pikepdf uses qpdf to decrypt and rewrite PDF objects. Pages are not printed or rasterized.

In [ ]:
%pip -q install 'pikepdf>=9.11,<10'

## Upload one PDF

In [ ]:
from google.colab import files
from pathlib import Path
import getpass
import pikepdf

uploaded = files.upload()
pdf_names = [name for name in uploaded if name.lower().endswith('.pdf')]
if len(pdf_names) != 1:
    raise ValueError('Upload exactly one PDF file.')

input_name = pdf_names[0]
input_path = Path('/content') / input_name
output_name = f'{input_path.stem}_unlocked.pdf'
output_path = Path('/content') / output_name
password = getpass.getpass('PDF password (input is hidden): ')

## Decrypt, verify, and download

A successful output must reopen with an empty password and have the same page count as the input. If the password is wrong or the file is unsupported, the notebook reports an error without producing a download.

In [ ]:
try:
    with pikepdf.open(input_path, password=password) as source:
        page_count = len(source.pages)
        source.save(output_path, encryption=False)

    with pikepdf.open(output_path) as verified:
        if len(verified.pages) != page_count:
            raise RuntimeError('Page-count verification failed.')

    input_bytes = input_path.stat().st_size
    output_bytes = output_path.stat().st_size
    print(f'Pages: {page_count}')
    print(f'Original size: {input_bytes:,} bytes')
    print(f'Unlocked copy: {output_bytes:,} bytes')
    files.download(str(output_path))
except pikepdf.PasswordError:
    output_path.unlink(missing_ok=True)
    print('The password was not accepted. No output was created.')
except (pikepdf.PdfError, OSError, RuntimeError) as error:
    output_path.unlink(missing_ok=True)
    print(f'Could not create a verified PDF: {type(error).__name__}. No output was downloaded.')
finally:
    password = None

## Optional cleanup

Run this cell after the download completes to remove the source and output files from the current runtime. Colab runtimes are temporary, but this explicit step removes the files earlier.

In [ ]:
for path in (input_path, output_path):
    if path.exists():
        path.unlink()
uploaded.clear()
print('Temporary PDF files removed from this runtime.')